# Intro to Data Science (journal 10)
by Asad Ali, 03-134202-013

## Objectives
###     Time-series Analysis & Forecasting
* Understand time series analysis and its applications.
* Read and Understand the Time Series Dataset.
* Preprocess and visualize the dataset.
* Build time series classification model.
* Make predictions on the test set.

**Dataset:** AirPassengers.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
plt.style.use('fivethirtyeight')
sns.set_palette('tab10')

In [ ]:
# loading dataset
data = 'AirPassengers.csv'
# since the Datetime is in object format
base = pd.read_csv(data, parse_dates=['Month'], index_col='Month')
base.rename(columns={'#Passengers':'Passengers'}, inplace=True)


In [ ]:
# Analysis of DS
cols = base.columns.tolist()
print(cols)
base.info()

In [ ]:
# information
print(base.head(2))
base.info()
base.describe()

In [ ]:
# Feature Engineering
def add_ts_features(df, lags=[1,12,24], roll_windows=[3,6,12]):
    df = df.copy()
    for lag in lags:
        df[f'lag_{lag}'] = df['Passengers'].shift(lag)
    # rolling means & stds
    for w in roll_windows:
        df[f'roll_mean_{w}'] = df['Passengers'].shift(1).rolling(window=w).mean()
        df[f'roll_std_{w}']  = df['Passengers'].shift(1).rolling(window=w).std()
    # drop NaNs created by shifts
    df = df.dropna()
    return df


p_ds = add_ts_features(base)
p_ds[['Passengers'] + [c for c in p_ds.columns if 'lag_' in c or 'roll_' in c]].head()
print(p_ds.head(2))

In [ ]:
# 4. Visualizations
# 4a. full time-series boxplot
plt.plot(p_ds['Passengers'], marker='.', alpha=0.8)
plt.title('Monthly Air Passengers (1949–1960)')
plt.ylabel('Passengers')
plt.xlabel('Date')
plt.show()

# 4b. Boxplot: Passengers by month to highlight seasonality :contentReference[oaicite:6]{index=6}:contentReference[oaicite:7]{index=7}
plt.figure(figsize=(10,6))
sns.boxplot(data=p_ds, x='month', y='Passengers')
plt.title('Passengers Distribution by Month')
plt.xlabel('Month')
plt.ylabel('Passengers')
plt.show()

# 4c. Boxplot: Passengers by year to show trend over time
plt.figure(figsize=(12,6))
sns.boxplot(data=p_ds, x='year', y='Passengers')
plt.title('Passengers Distribution by Year')
plt.xlabel('Year')
plt.ylabel('Passengers')
plt.xticks(rotation=45)
plt.show()

# 4d. Seasonal decomposition chart for trend/seasonality/residuals
decomp = seasonal_decompose(p_ds['Passengers'], model='additive', period=12)
fig = decomp.plot()
fig.set_size_inches(14,8)
plt.suptitle('Seasonal Decompose of Air Passengers', y=1.02)
plt.show()

In [ ]:
# 1. Train/Test Split (time‑based)
# --------------------------------

from sklearn.model_selection import TimeSeriesSplit 

# Prepare data
X = p_ds.drop('Passengers', axis=1)
y = p_ds['Passengers']
split_date = '1958-01-01'
X_train = X.loc[:split_date];  X_test = X.loc[split_date:]
y_train = y.loc[:split_date];  y_test = y.loc[split_date:]

tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
# 2. Import Models & Metrics
# --------------------------
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV 
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform

In [ ]:
# 3. Parameter distributions
# --------------------

rf_dist = {
    'n_estimators': randint(50, 300),
    'max_depth':    randint(5, 30),
    'max_features': uniform(0.5, 0.5)
}
xgb_dist = {
    'n_estimators':    randint(50, 300),
    'max_depth':       randint(3, 15),
    'learning_rate':   uniform(0.01, 0.3),
    'subsample':       uniform(0.6, 0.4),
    'colsample_bytree':uniform(0.6, 0.4)
}

# RF search
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    rf_dist,
    n_iter=50,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_search.fit(X_train, y_train)

# XGB search
xgb_search = RandomizedSearchCV(
    XGBRegressor(objective='reg:squarederror', random_state=42),
    xgb_dist,
    n_iter=50,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
xgb_search.fit(X_train, y_train)

In [ ]:
# 4. Predict
# ----------

best_rf  = rf_search.best_estimator_
best_xgb = xgb_search.best_estimator_

print("RF best params:",  rf_search.best_params_)
print("XGB best params:", xgb_search.best_params_)

y_pred_rf  = best_rf.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)

In [ ]:
# 5. Evaluate & Compare
# ----------------------
def eval_metrics(y_true, y_pred):
    return {
        'RMSE': mean_squared_error(y_true, y_pred ) **0.5,
        'MAE' : mean_absolute_error(y_true, y_pred),
        'R2'  : r2_score(y_true, y_pred)
    }

metrics_rf  = eval_metrics(y_test, y_pred_rf)
metrics_xgb = eval_metrics(y_test, y_pred_xgb)

import pandas as pd
results = pd.DataFrame([metrics_rf, metrics_xgb],
                       index=['RandomForest', 'XGBoost'])
print("\nModel performance on test set:")
print(results)

In [ ]:
# save best pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
import joblib

pipe_rf = Pipeline([
    ('ts_feat', FunctionTransformer(lambda d: add_ts_features(d))),
    ('model',    best_rf)
])
joblib.dump(pipe_rf, 'airpass_rf_pipeline.pkl')


In [ ]:
# 6. Plot Actual vs. Predicted
# -----------------------------
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(y_test.index, y_test,  label='Actual', marker='o', linestyle='-')
plt.plot(y_test.index, y_pred_rf,  label='RF Pred',  alpha=0.8)
plt.plot(y_test.index, y_pred_xgb, label='XGB Pred', alpha=0.8)
plt.title('Actual vs. Predicted Air Passengers')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.show()

In [ ]:
# 7. Note your findings
# ---------------------
print("\nFindings:")
print(f"- RandomForest:  RMSE={metrics_rf['RMSE']:.2f}, MAE={metrics_rf['MAE']:.2f}, R2={metrics_rf['R2']:.3f}")
print(f"- XGBoost:       RMSE={metrics_xgb['RMSE']:.2f}, MAE={metrics_xgb['MAE']:.2f}, R2={metrics_xgb['R2']:.3f}")
print("""
Observations:
1. Which model has lower RMSE/MAE?
2. Do both capture the overall upward trend?
3. Are there periods (e.g., peaks in 1958–1960) where one outperforms the other?
4. Based on R², how well does each explain the variance?
""")